In [1]:
import duckdb

parquet_file = "C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v3.parquet"
con = duckdb.connect()

diagnostic_df = con.execute(f"""
    SELECT 
        is_all_null,
        was_failed,
        COUNT(*) as serial_count,
        AVG(total_days) as avg_record_days,
        MIN(first_date) as min_date,
        MAX(last_date) as max_date
    FROM (
        SELECT 
            -- '_'를 기준으로 첫 번째 파트만 가져와서 진짜 S/N 추출
            split_part(serial_number, '_', 1) as base_serial,
            MAX(failure::INT) as was_failed,
            (COUNT(smart_241_raw) = 0) as is_all_null,
            COUNT(*) as total_days,
            MIN(date) as first_date,
            MAX(date) as last_date
        FROM read_parquet('{parquet_file}')
        GROUP BY base_serial  -- 진짜 시리얼 넘버로 그룹핑!
    )
    GROUP BY is_all_null, was_failed
    ORDER BY is_all_null DESC, was_failed DESC
""").fetchdf()

con.close()

print("=== 시리얼 넘버 접미사 제거 후 결과 ===")
print(diagnostic_df)

import duckdb

parquet_file = "C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v3.parquet"
con = duckdb.connect()

# 1. 접미사를 제거한 진짜 시리얼 넘버별로 센서 유무 체크
ghost_query = f"""
    SELECT 
        base_serial,
        MAX(failure::INT) as was_failed,
        COUNT(*) as total_rows
    FROM (
        SELECT 
            split_part(serial_number, '_', 1) as base_serial,
            smart_241_raw,
            failure
        FROM read_parquet('{parquet_file}')
    )
    GROUP BY base_serial
    HAVING COUNT(smart_241_raw) = 0  -- 모든 행이 NULL인 개체만 타겟팅
"""

ghost_df = con.execute(ghost_query).fetchdf()
con.close()

# 2. 결과 출력 및 리스트 추출
ghost_serials = ghost_df['base_serial'].unique().tolist()

print(f"=== 👻 접미사 제거 후 추출된 '유령 개체' 보고서 ===")
print(f"총 추출된 유령 개체 수: {len(ghost_serials)}대")

if len(ghost_serials) > 0:
    print(f"\n[상태 요약]")
    print(ghost_df['was_failed'].value_counts().rename({0: '정상(Healthy)', 1: '고장(Failed)'}))
    
    print("\n[추출된 시리얼 넘버 리스트]")
    print(ghost_serials)
    
    # 필요시 파일이나 텍스트로 저장할 수 있도록 준비
    # with open("ghost_serials.txt", "w") as f:
    #     f.write("\n".join(ghost_serials))
else:
    print("\n✅ 모든 행이 비어있는 개체가 더 이상 발견되지 않습니다.")

=== 시리얼 넘버 접미사 제거 후 결과 ===
   is_all_null  was_failed  serial_count  avg_record_days   min_date  \
0        False           1          5667      1316.625022 2013-05-10   
1        False           0         31269      2310.046564 2013-05-10   

    max_date  
0 2024-08-27  
1 2025-03-13  
=== 👻 접미사 제거 후 추출된 '유령 개체' 보고서 ===
총 추출된 유령 개체 수: 0대

✅ 모든 행이 비어있는 개체가 더 이상 발견되지 않습니다.


In [1]:
import duckdb
from pathlib import Path
from tqdm.auto import tqdm
import gc # 파이썬 메모리 강제 청소기

def convert_csv_to_parquet_duckdb_ultimate():
    src_dir = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\raw_data_csv"
    dest_dir = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\raw_data_parquet"
    
    src_path = Path(src_dir)
    dest_path = Path(dest_dir)
    dest_path.mkdir(parents=True, exist_ok=True)
    
    print("CSV 파일 탐색 및 월별 분류 중...")
    csv_files = list(src_path.rglob("*.csv"))
    
    monthly_files = {}
    for file_path in csv_files:
        month = file_path.stem[:7]
        if month not in monthly_files:
            monthly_files[month] = []
        monthly_files[month].append(str(file_path).replace('\\', '/'))
        
    print(f"총 {len(monthly_files)}개월 치의 데이터를 DuckDB를 이용해 병합합니다.\n")
    
    for month, files in tqdm(monthly_files.items(), desc="DuckDB 병합 (강제 램 관리 모드)"):
        output_filepath = dest_path / f"{month}.parquet"
        
        # 이미 변환된 파케이 파일 건너뛰기
        if output_filepath.exists():
            continue
            
        files.sort()
        output_str = str(output_filepath).replace('\\', '/')
        quoted_files = [f"'{f}'" for f in files]
        files_array_str = "[" + ", ".join(quoted_files) + "]"
        
        con = duckdb.connect()
        
        try:
            # 🌟 [초강력 조치 1] DuckDB가 사용할 수 있는 최대 램 상한선을 8GB로 막아버림!
            # 이렇게 하면 램이 꽉 차기 전에 알아서 내부 데이터를 버리고 비우며 작동합니다.
            con.execute("PRAGMA memory_limit='8GB';")
            
            query = f"""
                COPY (
                    SELECT * FROM read_csv(
                        {files_array_str}, 
                        union_by_name=true, 
                        ignore_errors=true,
                        types={{'date': 'VARCHAR', 'serial_number': 'VARCHAR', 'model': 'VARCHAR'}}
                    )
                ) TO '{output_str}' (FORMAT PARQUET);
            """
            con.execute(query)
            
        except Exception as e:
            print(f"[{month}] 변환 중 오류: {e}")
            
        finally:
            con.close()
            
            # 🌟 [초강력 조치 2] 한 바퀴 돌 때마다 메모리 청소부를 강제로 부름!
            # 이전 찌꺼기를 OS 레벨에서 완전히 즉시 반환하도록 멱살을 잡습니다.
            gc.collect()
            
    print("\n🎉 모든 변환이 안전하게 완료되었습니다!")

# 실행
if __name__ == "__main__":
    convert_csv_to_parquet_duckdb_ultimate()


CSV 파일 탐색 및 월별 분류 중...
총 147개월 치의 데이터를 DuckDB를 이용해 병합합니다.



DuckDB 병합 (강제 램 관리 모드):   0%|          | 0/147 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


🎉 모든 변환이 안전하게 완료되었습니다!


In [4]:
import duckdb
import pandas as pd

# 1. DuckDB 연결 생성 (메모리 모드)
con = duckdb.connect()

# 2. 전체 Parquet 파일을 읽어서 HDD 모델별 통계를 집계하는 강력한 SQL 쿼리
query = """
SELECT 
    model AS "모델",
    COUNT(*) AS "행 수",
    COUNT(DISTINCT serial_number) AS "개체 수",
    COUNT(DISTINCT CASE WHEN failure = 1 THEN serial_number END) AS "고장 개체 수",
    ROUND(CAST(COUNT(DISTINCT CASE WHEN failure = 1 THEN serial_number END) AS DOUBLE) / 
          CAST(COUNT(DISTINCT serial_number) AS DOUBLE) * 100, 4) AS "고장률 (%)"
FROM read_parquet('C:/Workspace/06_ML_projdect/26_1_COIN/data/raw_data_parquet/*.parquet')
GROUP BY model
ORDER BY "개체 수" DESC
LIMIT 20
"""

print("전체 데이터에서 모델별 통계 집계를 시작합니다. (수십 초가량 소요될 수 있습니다)")

# 3. 쿼리 실행 후 결과를 Pandas DataFrame으로 가져오기
model_stats_df = con.execute(query).fetchdf()

# 4. 결과 출력
display(model_stats_df.head(50))


전체 데이터에서 모델별 통계 집계를 시작합니다. (수십 초가량 소요될 수 있습니다)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,모델,행 수,개체 수,고장 개체 수,고장률 (%)
0,TOSHIBA MG08ACA16TA,27000887,40996,858,2.0929
1,TOSHIBA MG07ACA14TA,64805775,39387,1826,4.6360
2,ST12000NM0007,37055961,38843,2262,5.8234
3,WDC WUH722222ALE6L4,10978372,37451,243,0.6488
4,ST4000DM000,80454439,37040,5790,15.6317
5,ST16000NM001G,34692793,34755,684,1.9681
6,WDC WUH721816ALE6L4,21126913,26597,214,0.8046
7,ST12000NM0008,37903847,21037,2093,9.9491
8,HGST HMS5C4040BLE640,41138908,16349,448,2.7402
9,ST8000NM0055,41512527,15680,2255,14.3814


# 초기 88대 특이 개체 제거
- C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet에서 특이 개체의 시리얼 넘버를 알아낸 뒤에
- 모든 데이터 파일에 대해 해당 시리얼넘버를 가지는 행 제거하여 덮어쓰기 

In [15]:
import pandas as pd
import glob
import os

# 1. 제거할 베이스 시리얼 넘버 리스트 (set으로 변환하면 탐색 속도가 훨씬 빨라집니다)
ghost_serials = set([
    'Z300X3QC', 'W3006DHC', 'Z300CP8K', 'Z300CQVJ', 'Z300WEYV', 'W300BBWH', 'Z300KLZW', 'Z300WCS9', 
    'W3009P3W', 'W3008ZF4', 'Z300H22Q', 'Z300V4A8', 'W300BWGP', 'Z300E468', 'Z300K9MX', 'W3002Q5P', 
    'W300E0ZX', 'W3006E69', 'Z3004PWK', 'W300B38G', 'W3004ZV7', 'Z300E5HE', 'W30045V0', 'Z300GYYC', 
    'Z300NK21', 'W300D3ST', 'W300CK0E', 'Z300KKJZ', 'Z300VLEX', 'W300AE18', 'Z3001HG6', 'Z300KA2G', 
    'Z300GZ3D', 'W30065L7', 'W300DZ80', 'Z300HQ7W', 'Z300XGFJ', 'Z300X6Y6', 'Z300GZ80', 'W300B0C2', 
    'W300D3RY', 'Z300PKT6', 'Z300WJB1', 'W30062AH', 'W300B65C', 'Z300WWF9', 'Z300MQ46', 'W300CTKT', 
    'Z300HRV7', 'W300BNZ6', 'Z300PLYE', 'W300AR1D', 'W300A3L1', 'Z300MQDR', 'Z300T55K', 'W300B2VQ', 
    'W300D7FS', 'Z3000ZV3', 'Z300GQ62', 'Z300VMFN', 'W300JGR6', 'W300EE8B', 'W300BWX0', 'Z300K10V', 
    'W300CVHL', 'W300AYZP', 'Z300PC6H', 'Z300GQJP', 'Z300NKM3', 'Z300GZA9', 'Z300WN9X', 'W300506D', 
    'W300B6F3', 'Z3001N58', 'Z30149EZ', 'Z300XGWG', 'W300BHMT', 'W3006KGC', 'W3004NYW', 'W300KKTF', 
    'Z300WJ59', 'Z300CW7T', 'Z300GPZ8', 'W30065HL', 'W300BN35', 'W300D5W1', 'W300D5ES', 'Z300VN49'
])

# 2. 당장 작업할 파일 리스트 (파일명만 정확히 적어주세요)
target_dir = r"C:\Workspace\06_ML_projdect\26_1_COIN\data"
parquet_files = ["ST4000DM000_v3.parquet"]
# parquet_files = glob.glob(os.path.join(target_dir, "*.parquet"))

for file_name in parquet_files:
    file_path = os.path.join(target_dir, file_name)
    
    if not os.path.exists(file_path):
        print(f"❌ 파일을 찾을 수 없습니다: {file_name}")
        continue

    print(f"⏳ {file_name} 정제 시작 ...")
    
    df = pd.read_parquet(file_path)
    
    # 최적화: apply lambda가 str.split보다 대용량에서 조금 더 빠를 때가 있습니다.
    mask = df['serial_number'].apply(lambda x: x.split('_')[0] in ghost_serials)
    
    df_cleaned = df[~mask].copy()
    df_cleaned.to_parquet(file_path, index=False)
    
    print(f"✅ {file_name} 완료! ({len(df) - len(df_cleaned):,}개 행 제거)")

print("\n✨ 선택한 파일들의 정제가 완료되었습니다.")

⏳ ST4000DM000_v3.parquet 정제 시작 ...
✅ ST4000DM000_v3.parquet 완료! (4,228개 행 제거)

✨ 선택한 파일들의 정제가 완료되었습니다.


In [6]:
import duckdb

# 1. 파일 경로 설정
file_path = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_raw.parquet"

# 2. DuckDB 쿼리: 개체별 처음/마지막 날짜 추출
query = f"""
SELECT 
    serial_number,
    MIN(CAST(date AS DATE)) AS first_date,
    MAX(CAST(date AS DATE)) AS last_date,
    DATE_DIFF('day', MIN(CAST(date AS DATE)), MAX(CAST(date AS DATE))) AS duration_days
FROM '{file_path}'
GROUP BY serial_number
HAVING COUNT(smart_187_raw) = 0
ORDER BY first_date
"""

# 3. 쿼리 실행
df_peculiar = duckdb.query(query).df()

# 4. 팩트 체크 및 전체 그룹의 Min/Max 날짜 계산
actual_count = len(df_peculiar)
global_first_date = df_peculiar['first_date'].min()
global_last_date = df_peculiar['last_date'].max()

print(f"✅ 팩트 체크: 실제 특이 개체 수는 정확히 '{actual_count}대' 입니다.")
print("=" * 60)
print(f"🔥 [노이즈 구간 정의] 이 {actual_count}대 집단의 전체 타임라인:")
print(f" - 가장 처음 데이터가 찍힌 날 : {global_first_date}")
print(f" - 가장 마지막으로 데이터가 찍힌 날 : {global_last_date}")
print("=" * 60)

# 개별 하드디스크의 상세 정보 (상위 5개만 확인)
print("\n[개별 하드디스크 생존 요약 (Top 5)]")
print(df_peculiar.head())

# 원하시면 전체 결과를 CSV로 저장
# df_peculiar.to_csv("noise_89_units_timeline.csv", index=False)

✅ 팩트 체크: 실제 특이 개체 수는 정확히 '89대' 입니다.
🔥 [노이즈 구간 정의] 이 89대 집단의 전체 타임라인:
 - 가장 처음 데이터가 찍힌 날 : 2013-05-10 00:00:00
 - 가장 마지막으로 데이터가 찍힌 날 : 2014-02-13 00:00:00

[개별 하드디스크 생존 요약 (Top 5)]
  serial_number first_date  last_date  duration_days
0      Z3001N58 2013-05-10 2014-01-07            242
1      W300BWGP 2013-06-27 2013-08-19             53
2      W300B0C2 2013-06-27 2013-12-30            186
3      W300EE8B 2013-06-27 2013-07-03              6
4      W300AE18 2013-06-27 2014-01-21            208


In [3]:
import duckdb
import pandas as pd

# 파일 경로
parquet_path = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet"

con = duckdb.connect()

print("🔍 184번 센서 결측치 패턴 추적 중... (잠시만 기다려주세요 🚀)\n")

# arg_min, arg_max를 사용해서 첫날과 마지막 날의 값을 추출하고 패턴을 분류하는 쿼리
query = f"""
WITH DriveStats AS (
    SELECT 
        serial_number,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN smart_184_raw IS NULL THEN 1 ELSE 0 END) AS null_count,
        -- 가동 첫날의 184번 센서값
        arg_min(smart_184_raw, date) AS first_val,
        -- 가동 마지막 날의 184번 센서값
        arg_max(smart_184_raw, date) AS last_val,
        MAX(CAST(failure AS INT)) AS is_failed
    FROM read_parquet('{parquet_path}')
    GROUP BY serial_number
),
PatternCategorized AS (
    SELECT 
        serial_number,
        total_rows,
        null_count,
        is_failed,
        CASE 
            WHEN null_count = 0 THEN '🟢 1. 결측 없음 (처음부터 끝까지 정상)'
            WHEN null_count = total_rows THEN '🔴 2. 평생 결측 (첫날부터 죽을 때까지 NULL)'
            WHEN first_val IS NULL AND last_val IS NOT NULL THEN '🟡 3. 초기 결측 후 복구 (초반에 비어있다가 나중에 기록됨)'
            WHEN first_val IS NOT NULL AND null_count > 0 THEN '🟠 4. 중간/후반 결측 (초반엔 있었는데 중간에 끊김)'
            ELSE '⚪ 5. 기타 이상 패턴'
        END AS missing_pattern
    FROM DriveStats
)
SELECT 
    missing_pattern,
    COUNT(*) AS drive_count,
    ROUND(AVG(total_rows), 1) AS avg_lifespan_days,
    SUM(is_failed) AS fail_count,
    ROUND(AVG(is_failed) * 100, 2) AS fail_rate_percent
FROM PatternCategorized
GROUP BY missing_pattern
ORDER BY missing_pattern;
"""

df_pattern = con.execute(query).fetchdf()

# --- 🎨 출력 포맷 ---
print("="*75)
print(" 📊 [ST4000DM000] 184번 결측치 생애 주기 패턴 리포트")
print("="*75)

for _, row in df_pattern.iterrows():
    print(f"{row['missing_pattern']}")
    print(f"  - 해당 하드 수 : {int(row['drive_count']):,}대")
    print(f"  - 평균 수명    : {row['avg_lifespan_days']:,.1f}일")
    print(f"  - 고장 발생(률): {int(row['fail_count']):,}대 ({row['fail_rate_percent']}%)")
    print("-" * 75)

con.close()

🔍 184번 센서 결측치 패턴 추적 중... (잠시만 기다려주세요 🚀)

 📊 [ST4000DM000] 184번 결측치 생애 주기 패턴 리포트
🔴 2. 평생 결측 (첫날부터 죽을 때까지 NULL)
  - 해당 하드 수 : 2,457대
  - 평균 수명    : 23.4일
  - 고장 발생(률): 0대 (0.0%)
---------------------------------------------------------------------------
🟠 4. 중간/후반 결측 (초반엔 있었는데 중간에 끊김)
  - 해당 하드 수 : 5,807대
  - 평균 수명    : 842.7일
  - 고장 발생(률): 413대 (7.11%)
---------------------------------------------------------------------------
🟢 1. 결측 없음 (처음부터 끝까지 정상)
  - 해당 하드 수 : 80,110대
  - 평균 수명    : 933.0일
  - 고장 발생(률): 5,254대 (6.56%)
---------------------------------------------------------------------------


In [8]:
import duckdb
import pandas as pd

# 파일 경로
parquet_path = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet"

con = duckdb.connect()

print("🌍 전역적(Global) 184번 센서 결측치 발생 타임라인 분석 중... 🚀\n")

# 년-월 단위로 그룹핑해서 널(NaN)의 개수와 비율을 뽑아내는 쿼리
query = f"""
SELECT 
    strftime('%Y-%m', date) AS year_month,
    COUNT(*) AS total_records,
    SUM(CASE WHEN smart_184_raw IS NULL THEN 1 ELSE 0 END) AS null_count,
    ROUND(SUM(CASE WHEN smart_184_raw IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS null_ratio_percent
FROM read_parquet('{parquet_path}')
GROUP BY year_month
HAVING SUM(CASE WHEN smart_184_raw IS NULL THEN 1 ELSE 0 END) > 0
ORDER BY year_month;
"""

df = con.execute(query).fetchdf()

print("="*70)
print(" 🌍 [전역 분석] 184번 센서 결측치 월별 발생 트렌드")
print("="*70)
if df.empty:
    print("결측치가 존재하지 않습니다.")
else:
    print(df.to_string(index=False))
print("="*70)

con.close()

🌍 전역적(Global) 184번 센서 결측치 발생 타임라인 분석 중... 🚀

 🌍 [전역 분석] 184번 센서 결측치 월별 발생 트렌드
year_month  total_records  null_count  null_ratio_percent
   2013-05            968       968.0              100.00
   2013-06           3524      3524.0              100.00
   2013-07          26106     26106.0              100.00
   2013-08          19023     19023.0              100.00
   2013-09          17227     17227.0              100.00
   2013-10          60893     60893.0              100.00
   2013-11         119878    119878.0              100.00
   2013-12         145490    145490.0              100.00
   2014-01         174022    173033.0               99.43
   2014-02         192227     73046.0               38.00
   2014-03         252919       360.0                0.14
   2019-06         585793        20.0                0.00
   2019-07         604362        10.0                0.00
   2021-10         578389        37.0                0.01


In [7]:
import duckdb
import pandas as pd

# 파일 경로
parquet_path = r"C:\Workspace\06_ML_projdect\26_1_COIN\data\ST4000DM000_v3.parquet"
CUTOFF_DATE = '2014-03-01'

con = duckdb.connect()

print(f"📊 {CUTOFF_DATE} 기준, 데이터 삭제 전략별 손실률 비교 분석 중...")

query = f"""
WITH RawData AS (
    SELECT 
        regexp_replace(serial_number, '_[0-9]+$', '') AS base_serial,
        date,
        CAST(failure AS INT) as is_failed
    FROM read_parquet('{parquet_path}')
),
DriveInfo AS (
    -- 개체별 최초 가동일 확인
    SELECT 
        base_serial,
        MIN(date) as first_date
    FROM RawData
    GROUP BY 1
),
-- [전략 1] 행 단위 삭제
Strategy1 AS (
    SELECT 
        COUNT(DISTINCT base_serial) as drive_count,
        COUNT(*) as row_count,
        SUM(is_failed) as fail_count
    FROM RawData
    WHERE date >= '{CUTOFF_DATE}'
),
-- [전략 3] 세대 단위 삭제 (JOIN 시 컬럼명 앞에 d. 또는 r.을 붙여 명시)
Strategy3 AS (
    SELECT 
        COUNT(DISTINCT d.base_serial) as drive_count,
        COUNT(r.date) as row_count,
        SUM(r.is_failed) as fail_count
    FROM DriveInfo d
    JOIN RawData r ON d.base_serial = r.base_serial
    WHERE d.first_date >= '{CUTOFF_DATE}'
),
-- [원본] 전체 데이터
Total AS (
    SELECT 
        COUNT(DISTINCT base_serial) as drive_count,
        COUNT(*) as row_count,
        SUM(is_failed) as fail_count
    FROM RawData
)
-- 결과 합치기
SELECT '원본 데이터' as strategy, * FROM Total
UNION ALL
SELECT '방법 1 (행 단위 삭제)', * FROM Strategy1
UNION ALL
SELECT '방법 3 (세대 단위 삭제)', * FROM Strategy3
"""

df_comp = con.execute(query).fetchdf()

# --- 🎨 비교 리포트 출력 ---
print("\n" + "="*80)
print(f" 📋 [최종 견적서] 데이터 정제 전략 비교 (기준일: {CUTOFF_DATE})")
print("="*80)

base = df_comp.iloc[0]
for i in range(len(df_comp)):
    row = df_comp.iloc[i]
    print(f"[{row['strategy']}]")
    print(f"  - 하드디스크 수 : {int(row['drive_count']):>8,}대 (손실률: {(1 - row['drive_count']/base['drive_count'])*100:>5.1f}%)")
    print(f"  - 데이터 행 수  : {int(row['row_count']):>8,}행 (손실률: {(1 - row['row_count']/base['row_count'])*100:>5.1f}%)")
    print(f"  - 고장 타겟  : {int(row['fail_count']):>8,}건 (손실률: {(1 - row['fail_count']/base['fail_count'])*100:>5.1f}%)")
    print("-" * 80)

con.close()

📊 2014-03-01 기준, 데이터 삭제 전략별 손실률 비교 분석 중...

 📋 [최종 견적서] 데이터 정제 전략 비교 (기준일: 2014-03-01)
[원본 데이터]
  - 하드디스크 수 :   36,936대 (손실률:   0.0%)
  - 데이터 행 수  : 79,694,160행 (손실률:   0.0%)
  - 고장 타겟  :   55,965건 (손실률:   0.0%)
--------------------------------------------------------------------------------
[방법 1 (행 단위 삭제)]
  - 하드디스크 수 :   36,926대 (손실률:   0.0%)
  - 데이터 행 수  : 78,934,802행 (손실률:   1.0%)
  - 고장 타겟  :   55,873건 (손실률:   0.2%)
--------------------------------------------------------------------------------
[방법 3 (세대 단위 삭제)]
  - 하드디스크 수 :   28,984대 (손실률:  21.5%)
  - 데이터 행 수  : 67,686,668행 (손실률:  15.1%)
  - 고장 타겟  :   43,715건 (손실률:  21.9%)
--------------------------------------------------------------------------------
